In [ ]:
#| default_exp search

In [ ]:
#| export
from fastcore.all import patch
from fastlite import Database
from litesearch.core import rrf_all, sql_in

@patch
def graph_search(self:Database, q:str, emb:bytes, store:str='store', prefix:str=None,
                 limit:int=10, seedk:int=8, graph_w:float=1.0):
    "Hybrid seed, walk one relation hop to neighbour entities, pull their chunks, RRF-fuse. LLM-built typed graph."
    g = self.get_graph(store, prefix)
    base = self.search(q, emb, columns=['id','node_id'], limit=50, table_name=store) or []
    seeds = [h['id'] for h in base[:seedk]]
    if not seeds: return base[:limit]
    sent = {m['entity_id'] for m in g.mentions(select='entity_id', where=sql_in('chunk_id', seeds))}
    nbr = {e['dst'] for e in g.edges(select='dst', where=sql_in('src', list(sent)))} if sent else set()
    if not nbr: return base[:limit]
    leg_ids = list(dict.fromkeys(m['chunk_id'] for m in g.mentions(select='chunk_id', where=sql_in('entity_id', list(nbr)))))
    node = {r['id']:r['node_id'] for r in self.t[store](select='id, node_id', where=sql_in('id', leg_ids))} if leg_ids else {}
    return rrf_all([base, [dict(id=i, node_id=node.get(i)) for i in leg_ids]], k=60, limit=limit, id_key='id', weights=[1.0, graph_w])

def graph_stats(db, store='store', prefix=None):
    "Row counts and the highest-degree entities of a built graph."
    g = db.get_graph(store, prefix)
    deg = {}
    for e in g.edges(select='src, dst'):
        deg[e['src']] = deg.get(e['src'],0)+1; deg[e['dst']] = deg.get(e['dst'],0)+1
    top = sorted(deg.items(), key=lambda kv: -kv[1])[:10]
    name = {r['id']:r['content'] for r in g.entities(select='id, content')}
    return dict(entities=g.entities.count, mentions=g.mentions.count, edges=g.edges.count,
                top_degree=[dict(entity=name.get(i,i), degree=d) for i,d in top])

In [ ]:
#| hide
import numpy as np, hashlib
from litesearch import database
from vruksha.build import build_graph
def _femb(ts): return [np.frombuffer(hashlib.sha256(t.encode()).digest()[:16], dtype=np.uint8).astype(np.float16)/255 for t in ts]
class _Chat:
    def oneshot(self, u, **k):
        return ('{"summary":"s","entities":[{"name":"bm25","type":"method"}],'
                '"relations":[{"src":"bm25","rel":"cites","dst":"robertson"}]}')
db = database(); st = db.get_store(name='store', hash=True, ann=True, node_id=str)
rows = [dict(content=f'passage {i} discussing bm25 and robertson', node_id=f'd#{i}') for i in range(3)]
for r,v in zip(rows, _femb([r['content'] for r in rows])): r['embedding'] = v.tobytes()
st.insert_all(rows, upsert=True, hash_id='id', hash_id_columns=['content']); st.rebuild_index()
build_graph(db, _Chat(), _femb, store='store', batch=8)
hits = db.graph_search('bm25 robertson', _femb(['bm25 robertson'])[0].tobytes(), limit=5)
assert isinstance(hits, list)
assert graph_stats(db,'store')['entities'] >= 1